In [1]:
import findspark 
findspark.init()

In [2]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

#### Tạo map (UserID -> Age Group).

In [4]:
users = sc.textFile("file:///C:/Users/Dell/IE212/Lab3/users.txt")

In [5]:
def age_group (age):
    age = int (age)
    if age < 18:
        return "0-17"
    elif age <= 25:
        return "18-25"
    elif age <= 35:
        return "26-35"
    elif age <= 45:
        return "36-45"
    elif age <= 55:
        return "46-55"
    else:
        return "56+"
users_rdd = users.map(lambda x: x.split(",")).map(lambda x: ((x[0]), age_group(x[2])))
users_rdd.take(5)

[('1', '26-35'),
 ('2', '26-35'),
 ('3', '36-45'),
 ('4', '18-25'),
 ('5', '26-35')]

In [6]:
ratings1 = sc.textFile("file:///C:/Users/Dell/IE212/Lab3/ratings_1.txt")
ratings2 = sc.textFile("file:///C:/Users/Dell/IE212/Lab3/ratings_2.txt")
ratings = ratings1.union(ratings2)
ratings_rdd = ratings.map(lambda x: x.split(",")).map(lambda x: (x[0], (x[1], float(x[2]))))
ratings_rdd.take(5)                                                

[('7', ('1020', 4.5)),
 ('23', ('1015', 3.5)),
 ('45', ('1030', 4.0)),
 ('12', ('1047', 3.0)),
 ('38', ('1012', 4.5))]

In [7]:
users_rating = ratings_rdd.join(users_rdd)
users_rating.take(5)

[('34', (('1030', 3.0), '36-45')),
 ('34', (('1039', 4.0), '36-45')),
 ('34', (('1025', 3.5), '36-45')),
 ('34', (('1037', 3.0), '36-45')),
 ('4', (('1037', 4.0), '18-25'))]

In [8]:
movie_age_rating = users_rating.map(lambda x : ((x[1][0][0], x[1][1]), x[1][0][1]))
movie_age_rating.take(5)

[(('1030', '36-45'), 3.0),
 (('1039', '36-45'), 4.0),
 (('1025', '36-45'), 3.5),
 (('1037', '36-45'), 3.0),
 (('1037', '18-25'), 4.0)]

In [9]:
sum_rdd = movie_age_rating.reduceByKey(lambda a,b : a + b)
sum_rdd.take(5)

[(('1047', '26-35'), 12.0),
 (('1010', '36-45'), 33.5),
 (('1010', '26-35'), 18.0),
 (('1050', '26-35'), 34.0),
 (('1028', '46-55'), 7.0)]

In [10]:
count_rdd = movie_age_rating.mapValues(lambda x: 1).reduceByKey(lambda a,b : a + b)
count_rdd.take(5)

[(('1047', '26-35'), 3),
 (('1010', '36-45'), 10),
 (('1010', '26-35'), 5),
 (('1050', '26-35'), 10),
 (('1028', '46-55'), 2)]

In [11]:
average_rdd = sum_rdd.join(count_rdd).mapValues(lambda x: x[0] / x[1])
average_rdd = average_rdd.sortByKey()
for row in average_rdd.collect():
    print(row)

(('1010', '26-35'), 3.6)
(('1010', '36-45'), 3.35)
(('1010', '46-55'), 3.5)
(('1012', '26-35'), 4.5)
(('1012', '36-45'), 3.5)
(('1013', '18-25'), 4.0)
(('1013', '26-35'), 3.7142857142857144)
(('1013', '36-45'), 4.25)
(('1015', '18-25'), 4.5)
(('1015', '26-35'), 4.0)
(('1015', '36-45'), 4.5)
(('1015', '46-55'), 4.5)
(('1020', '18-25'), 3.5)
(('1020', '26-35'), 3.5833333333333335)
(('1020', '36-45'), 3.8125)
(('1020', '46-55'), 4.0)
(('1020', '56+'), 3.0)
(('1025', '18-25'), 4.5)
(('1025', '26-35'), 4.1)
(('1025', '36-45'), 4.0)
(('1025', '46-55'), 4.166666666666667)
(('1025', '56+'), 3.75)
(('1028', '26-35'), 3.5)
(('1028', '36-45'), 3.5)
(('1028', '46-55'), 3.5)
(('1030', '26-35'), 3.0)
(('1030', '36-45'), 3.0)
(('1030', '46-55'), 3.3333333333333335)
(('1037', '18-25'), 4.0)
(('1037', '26-35'), 4.0)
(('1037', '36-45'), 3.7222222222222223)
(('1037', '46-55'), 4.166666666666667)
(('1039', '26-35'), 3.8333333333333335)
(('1039', '36-45'), 3.857142857142857)
(('1039', '46-55'), 3.5)
(('104